In [ ]:
!pip -q install -U "transformers>=4.48.0" "peft>=0.14.0" "accelerate>=1.2.0" sentencepiece tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 43.2 MB/s eta 0:00:00


In [ ]:
# Fix torchao conflict for PEFT LoRA injection
!pip -q uninstall -y torchao
!pip -q install -U "torchao>=0.16.0"

import os
print("torchao updated. Restarting runtime...")
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/patcher_final_overnight_run/run/last_adapter"
EVAL_JSONL = "/content/drive/MyDrive/patch_patcher_v3/outputs/patcher_eval.jsonl"
REPO_DIR = "/content/airflow"

SMOKE_N = 20
MAX_NEW_TOKENS = 1000
N_CANDS = 5
REVISE_N = 2

print("MODEL_NAME:", MODEL_NAME)
print("ADAPTER_PATH:", ADAPTER_PATH)
print("EVAL_JSONL:", EVAL_JSONL)
print("REPO_DIR:", REPO_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL_NAME: Qwen/Qwen2.5-Coder-7B-Instruct
ADAPTER_PATH: /content/drive/MyDrive/patcher_final_overnight_run/run/last_adapter
EVAL_JSONL: /content/drive/MyDrive/patch_patcher_v3/outputs/patcher_eval.jsonl
REPO_DIR: /content/airflow


In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    print("Cloning Airflow...")
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/apache/airflow.git", REPO_DIR],
        check=True
    )

subprocess.run(["git", "-C", REPO_DIR, "fetch", "--unshallow"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--is-inside-work-tree"], check=True)

print("Repo ready:", REPO_DIR)

Repo ready: /content/airflow


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

assert os.path.isdir(ADAPTER_PATH), f"Missing adapter path: {ADAPTER_PATH}"
assert os.path.exists(EVAL_JSONL), f"Missing eval jsonl: {EVAL_JSONL}"

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ADAPTER_PATH).eval()
print("Loaded adapter successfully.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded adapter successfully.


In [ ]:
import json, re, subprocess
from typing import Any, Dict, List, Tuple

def clip_text(x: str, n: int) -> str:
    x = "" if x is None else str(x)
    x = x.replace("\r", "\n").strip()
    return x[:n]

def format_planner(pl: Dict[str, Any]) -> str:
    files = pl.get("target_files", []) or []
    funcs = pl.get("target_functions", []) or []
    lines = [
        "--- PLANNER DIRECTIVE ---",
        f"REQUIRES_CODE_CHANGE: {pl.get('requires_code_change', 'YES')}",
        f"CONFIDENCE: {pl.get('confidence', 'MEDIUM')}",
        f"REASON: {clip_text(pl.get('reason', ''), 260)}",
        "PLAN:",
        f"- Root cause: {clip_text(pl.get('root_cause', ''), 260)}",
        "- Target files:",
    ]
    lines.extend([f"  - {f}" for f in files])
    if funcs:
        lines.append("- Target functions:")
        lines.extend([f"  - {fn}" for fn in funcs[:16]])
    lines.append(f"- Test strategy: {clip_text(pl.get('test_strategy', ''), 220)}")
    return "\n".join(lines)

def format_treesitter(ts_ctx: Dict[str, Any], max_spans: int = 8) -> str:
    spans = ts_ctx.get("spans", []) or []
    spans = sorted(
        spans,
        key=lambda s: (not bool(s.get("touched_by_gold_patch", False)), s.get("start_line", 0)),
    )[:max_spans]
    out = ["--- TREE_SITTER_SPANS ---"]
    if not spans:
        out.append("None")
        return "\n".join(out)
    for s in spans:
        out.append(f"# FILE: {s.get('file', '')}")
        out.append(f"# SYMBOL: {s.get('symbol', '')} ({s.get('symbol_type', '')})")
        out.append(f"# RANGE: {s.get('start_line', 0)}-{s.get('end_line', 0)}")
        out.append(clip_text(s.get("code", ""), 2200))
        out.append("")
    return "\n".join(out).strip()

def format_graphrag(gr: Dict[str, Any]) -> str:
    out = ["--- HISTORICAL REPO IDIOMS (GraphRAG) ---"]
    cands = (gr.get("candidate_files_topk", []) or [])[:3]
    idioms = (gr.get("historical_idioms", []) or [])[:4]
    if cands:
        out.append("Top candidate files:")
        for c in cands:
            out.append(f"- {c.get('file', '')} (score={c.get('score', 0)})")
    else:
        out.append("Top candidate files: None")
    if idioms:
        out.append("Historical snippets:")
        for it in idioms:
            out.append(
                f"- [{it.get('file','')}] PR#{it.get('source_pr','')}: "
                f"{clip_text(it.get('snippet',''), 120)}"
            )
    else:
        out.append("Historical snippets: None")
    return "\n".join(out)

def format_constraints(c: Dict[str, Any]) -> str:
    allowed = c.get("allowed_files", []) or []
    lines = ["--- CONSTRAINTS ---", "Allowed files:"]
    lines.extend([f"  - {x}" for x in allowed])
    lines.append(f"forbid_new_files: {bool(c.get('forbid_new_files', True))}")
    lines.append(f"forbid_unrelated_refactors: {bool(c.get('forbid_unrelated_refactors', True))}")
    lines.append(f"output_format: {c.get('output_format', 'unified_diff_only')}")
    return "\n".join(lines)

def build_prompt(row: Dict[str, Any]) -> str:
    inp = row.get("input", {})
    return (
        "You are an expert patch generation model.\n"
        "Generate the minimal correct unified diff patch.\n"
        "Hard rules:\n"
        "- Touch only allowed files.\n"
        "- Prefer edits inside or adjacent to touched Tree-sitter spans.\n"
        "- Keep the patch minimal and avoid unrelated refactors/reformatting.\n"
        "- If uncertain, make the smallest safe change only.\n"
        "- Do NOT use markdown fences.\n"
        "- Output exactly one unified diff patch and then stop.\n"
        "- Output ONLY unified diff text starting with --- a/.\n\n"
        f"INSTRUCTION: {clip_text(inp.get('instruction', ''), 400)}\n\n"
        f"{format_planner(inp.get('planner_directive', {}))}\n\n"
        f"{format_constraints(inp.get('constraints', {}))}\n\n"
        f"{format_graphrag(inp.get('graphrag_context', {}))}\n\n"
        f"{format_treesitter(inp.get('treesitter_context', {}))}\n\n"
        "TASK: Produce the patch now.\n"
    )

def is_valid_unified_diff(text: str) -> bool:
    return text.startswith("--- a/") and ("\n+++ b/" in text) and ("\n@@" in text)

def parse_pred_files(diff_text: str) -> set:
    return set(re.findall(r"^\+\+\+ b/(.+)$", diff_text, flags=re.MULTILINE))

def edit_size(diff_text: str) -> int:
    adds = sum(1 for x in diff_text.splitlines() if x.startswith("+") and not x.startswith("+++"))
    dels = sum(1 for x in diff_text.splitlines() if x.startswith("-") and not x.startswith("---"))
    return adds + dels

def sanitize_diff_output(raw: str) -> str:
    if raw is None:
        return ""
    s = str(raw).strip()
    s = re.sub(r"```(?:diff|patch|text)?", "", s, flags=re.IGNORECASE)
    s = s.replace("```", "").strip()

    m = re.search(r"^--- a/.*$", s, flags=re.MULTILINE)
    if not m:
        return s
    s = s[m.start():].strip()
    lines = s.splitlines()

    while lines and lines[-1].strip() in {"---", "```", ""}:
        lines.pop()

    starts = [i for i, ln in enumerate(lines) if ln.startswith("--- a/")]
    if len(starts) >= 2:
        a, b = starts[0], starts[1]
        first_file = lines[a][6:].strip() if lines[a].startswith("--- a/") else None
        second_file = lines[b][6:].strip() if lines[b].startswith("--- a/") else None
        if first_file and second_file and first_file == second_file:
            lines = lines[:b]

    return "\n".join(lines).strip()

def parse_unified_files(diff_text: str):
    lines = diff_text.splitlines()
    files = []
    i = 0
    while i < len(lines):
        if not lines[i].startswith("--- a/"):
            i += 1
            continue
        old_path = lines[i][6:]
        i += 1
        if i >= len(lines) or not lines[i].startswith("+++ b/"):
            break
        new_path = lines[i][6:]
        i += 1
        body = []
        while i < len(lines) and not lines[i].startswith("--- a/"):
            body.append(lines[i]); i += 1
        files.append({"old": old_path, "new": new_path, "body_lines": body})
    return files

def parse_hunk_header_line(hline: str):
    m = re.match(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@", hline)
    if not m:
        return None
    return {
        "old_start": int(m.group(1)),
        "old_count": int(m.group(2) or "1"),
        "new_start": int(m.group(3)),
        "new_count": int(m.group(4) or "1"),
    }

def apply_patch_text_to_source(src: str, file_diff_text: str):
    src_lines = src.splitlines()
    out_lines, cursor = [], 0
    lines = file_diff_text.splitlines()
    i = 0
    while i < len(lines):
        if not lines[i].startswith("@@ "):
            i += 1; continue
        h = parse_hunk_header_line(lines[i])
        if h is None: return False, src
        old_idx = h["old_start"] - 1
        if old_idx < cursor or old_idx > len(src_lines): return False, src
        out_lines.extend(src_lines[cursor:old_idx])
        src_ptr = old_idx
        i += 1
        while i < len(lines) and not lines[i].startswith("@@ "):
            raw = lines[i]
            if raw.startswith("\\"):
                i += 1; continue
            if raw == "":
                return False, src
            tag, content = raw[0], raw[1:]
            if tag == " ":
                if src_ptr >= len(src_lines) or src_lines[src_ptr] != content: return False, src
                out_lines.append(src_lines[src_ptr]); src_ptr += 1
            elif tag == "-":
                if src_ptr >= len(src_lines) or src_lines[src_ptr] != content: return False, src
                src_ptr += 1
            elif tag == "+":
                out_lines.append(content)
            else:
                return False, src
            i += 1
        cursor = src_ptr
    out_lines.extend(src_lines[cursor:])
    patched = "\n".join(out_lines)
    if src.endswith("\n"): patched += "\n"
    return True, patched

def git_show_file_at_sha(repo_dir: str, sha: str, file_path: str):
    p = subprocess.run(
        ["git", "-C", repo_dir, "show", f"{sha}:{file_path}"],
        capture_output=True, text=True
    )
    if p.returncode != 0:
        return None, (p.stderr or "").strip()
    return p.stdout, ""

def diagnose_apply(diff_text: str, meta: Dict[str, Any]) -> Tuple[bool, str]:
    if not is_valid_unified_diff(diff_text):
        return False, "invalid_unified_diff_shape"
    base_sha = meta.get("base_sha")
    if not base_sha:
        return False, "missing_base_sha"
    files = parse_unified_files(diff_text)
    if not files:
        return False, "parse_unified_files_empty"
    for f in files:
        fp = f.get("old")
        if not fp: return False, "missing_old_path"
        src, err = git_show_file_at_sha(REPO_DIR, base_sha, fp)
        if src is None:
            return False, f"git_show_failed:{fp}:{err[:120]}"
        ok, _ = apply_patch_text_to_source(src, "\n".join(f.get("body_lines", [])))
        if not ok:
            return False, f"apply_failed:{fp}"
    return True, "ok"

def generate_n(prompt: str, n: int, max_new_tokens: int):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    cands = []
    for i in range(n):
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
                top_k=40,
                pad_token_id=tok.eos_token_id,
            )
        raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        pred = sanitize_diff_output(raw)
        cands.append({"candidate_idx": i, "pred": pred, "raw": raw})
    return cands

def score_candidate(pred: str, allowed_files: List[str], meta: Dict[str, Any]) -> Dict[str, Any]:
    valid = is_valid_unified_diff(pred)
    p_files = parse_pred_files(pred)
    allowed_ok = bool(p_files) and p_files.issubset(set(allowed_files))
    apply_ok, reason = diagnose_apply(pred, meta)
    size = edit_size(pred) if pred else 10**9
    return {
        "valid": valid,
        "allowed_ok": allowed_ok,
        "apply_ok": apply_ok,
        "reason": reason,
        "edit_size": size,
    }

def pick_best(rows):
    rows_sorted = sorted(
        rows,
        key=lambda x: (
            int(x["score"]["apply_ok"]),
            int(x["score"]["allowed_ok"]),
            int(x["score"]["valid"]),
            -x["score"]["edit_size"],
        ),
        reverse=True,
    )
    return rows_sorted[0], rows_sorted

In [ ]:
rows = []
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

eval_examples = []
for r in rows[:SMOKE_N]:
    eval_examples.append({
        "input": build_prompt(r),
        "output": r.get("output", {}).get("unified_diff", "").strip(),
        "meta": {
            "base_sha": r.get("meta", {}).get("base_sha"),
            "allowed_files": r.get("input", {}).get("constraints", {}).get("allowed_files", []) or [],
        },
    })

print("Loaded eval examples:", len(eval_examples))

Loaded eval examples: 20


In [ ]:
from tqdm import tqdm

def run_one_with_retry(ex):
    prompt = ex["input"]
    allowed_files = ex["meta"]["allowed_files"]
    meta = {"base_sha": ex["meta"]["base_sha"]}

    # Round 1
    cands = generate_n(prompt, n=N_CANDS, max_new_tokens=MAX_NEW_TOKENS)
    scored = []
    for c in cands:
        scored.append({
            "round": 1,
            "pred": c["pred"],
            "raw": c["raw"],
            "score": score_candidate(c["pred"], allowed_files, meta),
        })

    best, ranked = pick_best(scored)
    if best["score"]["apply_ok"]:
        return {"status": "APPLY_PASS", "best": best, "all": ranked, "revise_used": False}

    # Round 2 (one revise pass)
    revise_feedback = (
        "\n\nREVISION FEEDBACK:\n"
        f"Previous patch failed apply check: {best['score']['reason']}\n"
        "Regenerate ONLY unified diff.\n"
        "Do NOT use markdown fences.\n"
        "Output exactly one unified diff patch and then stop.\n"
        "Keep same intent and allowed files.\n"
        "Do not add unrelated edits.\n"
        "Preserve exact context lines around hunks.\n"
        "Prefer smallest possible change.\n"
    )
    revise_prompt = prompt + revise_feedback
    rcands = generate_n(revise_prompt, n=REVISE_N, max_new_tokens=MAX_NEW_TOKENS)

    rscored = []
    for c in rcands:
        rscored.append({
            "round": 2,
            "pred": c["pred"],
            "raw": c["raw"],
            "score": score_candidate(c["pred"], allowed_files, meta),
        })

    combined = scored + rscored
    best2, ranked2 = pick_best(combined)
    status = "APPLY_PASS" if best2["score"]["apply_ok"] else "DRAFT_ONLY"
    return {"status": status, "best": best2, "all": ranked2, "revise_used": True}

pass1 = 0
passN = 0
passNrev = 0
out_rows = []

for i, ex in enumerate(tqdm(eval_examples, desc="orchestrator-smoke")):
    # pass@1
    c1 = generate_n(ex["input"], n=1, max_new_tokens=MAX_NEW_TOKENS)[0]
    s1 = score_candidate(c1["pred"], ex["meta"]["allowed_files"], {"base_sha": ex["meta"]["base_sha"]})
    if s1["apply_ok"]:
        pass1 += 1

    # pass@N and pass@N+revise1
    result = run_one_with_retry(ex)
    round1_any = any(r["score"]["apply_ok"] for r in result["all"] if r["round"] == 1)
    if round1_any:
        passN += 1
    if result["status"] == "APPLY_PASS":
        passNrev += 1

    out_rows.append({
        "i": i,
        "status": result["status"],
        "revise_used": result["revise_used"],
        "best_reason": result["best"]["score"]["reason"],
        "best_apply_ok": result["best"]["score"]["apply_ok"],
        "best_valid": result["best"]["score"]["valid"],
        "best_allowed_ok": result["best"]["score"]["allowed_ok"],
        "best_edit_size": result["best"]["score"]["edit_size"],
    })

n = max(1, len(eval_examples))
summary = {
    "samples": n,
    "pass@1": pass1 / n,
    "pass@N": passN / n,
    "pass@N+revise1": passNrev / n,
    "counts": {"pass1": pass1, "passN": passN, "passNrev": passNrev},
}

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))
print("\n=== FIRST 10 ROWS ===")
print(json.dumps(out_rows[:10], indent=2))

orchestrator-smoke: 100%|██████████| 20/20 [2:38:12<00:00, 474.64s/it]


=== SUMMARY ===
{
  "samples": 20,
  "pass@1": 0.0,
  "pass@N": 0.0,
  "pass@N+revise1": 0.0,
  "counts": {
    "pass1": 0,
    "passN": 0,
    "passNrev": 0
  }
}

=== FIRST 10 ROWS ===
[
  {
    "i": 0,
    "status": "DRAFT_ONLY",
    "revise_used": true,
    "best_reason": "apply_failed:dev/README_RELEASE_PYTHON_CLIENT.md",
    "best_apply_ok": false,
    "best_valid": true,
    "best_allowed_ok": true,
    "best_edit_size": 2
  },
  {
    "i": 1,
    "status": "DRAFT_ONLY",
    "revise_used": true,
    "best_reason": "apply_failed:airflow/www/static/js/components/SourceTaskInstance.tsx",
    "best_apply_ok": false,
    "best_valid": true,
    "best_allowed_ok": true,
    "best_edit_size": 2
  },
  {
    "i": 2,
    "status": "DRAFT_ONLY",
    "revise_used": true,
    "best_reason": "apply_failed:airflow/www/static/js/utils/index.test.ts",
    "best_apply_ok": false,
    "best_valid": true,
    "best_allowed_ok": true,
    "best_edit_size": 8
  },
  {
    "i": 3,
    "status": "DRA